In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix
)

print("="*70)
print("PHASE 1 — CLEAN TEXT BASELINE MODEL")
print("="*70)

# 1. Load Original Dataset
df = pd.read_csv(r'data/fake_job_postings.csv')

# 2. Exact Stratified Split (80/20)
X = df.drop('fraudulent', axis=1)
y = df['fraudulent']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Total dataset: {len(df)} postings")
print(f"Train set    : {len(X_train)} samples ({y_train.sum()} fraudulent)")
print(f"Test set     : {len(X_test)} samples ({y_test.sum()} fraudulent)")

# 3. Extract and Combine Text Features
X_train_p1 = X_train.copy()
X_test_p1 = X_test.copy()

X_train_p1['description'] = X_train_p1['description'].fillna('')
X_train_p1['requirements'] = X_train_p1['requirements'].fillna('')
X_train_p1['text'] = X_train_p1['description'] + " " + X_train_p1['requirements']

X_test_p1['description'] = X_test_p1['description'].fillna('')
X_test_p1['requirements'] = X_test_p1['requirements'].fillna('')
X_test_p1['text'] = X_test_p1['description'] + " " + X_test_p1['requirements']

# 4. TF-IDF Vectorization (Fit strictly on Train)
print("\n⚙️ Vectorizing text with TF-IDF (max_features=3000, ngram_range=(1,2))...")
tfidf_p1 = TfidfVectorizer(
    max_features=3000,
    stop_words='english',
    ngram_range=(1, 2)
)

X_train_tfidf_p1 = tfidf_p1.fit_transform(X_train_p1['text'])
X_test_tfidf_p1 = tfidf_p1.transform(X_test_p1['text'])

# 5. Train Baseline Classifier
print("⚙️ Training Baseline Logistic Regression Classifier...")
lr_p1 = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)
lr_p1.fit(X_train_tfidf_p1, y_train)

# 6. Evaluation
y_pred_p1 = lr_p1.predict(X_test_tfidf_p1)
y_proba_p1 = lr_p1.predict_proba(X_test_tfidf_p1)[:, 1]

acc_p1 = accuracy_score(y_test, y_pred_p1)
prec_p1 = precision_score(y_test, y_pred_p1)
rec_p1 = recall_score(y_test, y_pred_p1)
f1_p1 = f1_score(y_test, y_pred_p1)
auc_p1 = roc_auc_score(y_test, y_proba_p1)

print("\n" + "="*70)
print("📊 PHASE 1 RESULTS: Baseline Performance")
print("="*70)
print(f"Accuracy  : {acc_p1:.4f}")
print(f"Precision : {prec_p1:.4f}")
print(f"Recall    : {rec_p1:.4f}  <-- Expected Baseline Recall ~85.55%")
print(f"F1-Score  : {f1_p1:.4f}")
print(f"ROC-AUC   : {auc_p1:.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_p1))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_p1))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix
)

print("="*70)
print("PHASE 2 — LLM TEXT EVASION ATTACK (INFERENCE-TIME)")
print("="*70)

# Path to local rewritten test fraud jobs
PATH_TEST_REWRITTEN = r'data/173_fraud_test_jobs_to_rewrite.csv'

# Helper function to safely read CSV with encoding fallback
def load_csv_safe(filepath):
    try:
        return pd.read_csv(filepath)
    except UnicodeDecodeError:
        return pd.read_csv(filepath, encoding='latin-1')

# 1. Load the 173 AI-rewritten test fraud jobs
test_rewritten_fraud = load_csv_safe(PATH_TEST_REWRITTEN)

# 2. Construct Attacked Test Data (Dataset C)
X_test_p2 = X_test.copy()

# Index by job_id for clean update
X_test_p2.set_index('job_id', inplace=True)
rewritten_test_df = test_rewritten_fraud.copy()
rewritten_test_df.set_index('job_id', inplace=True)

# Update ONLY description and requirements for the 173 test fraud jobs
X_test_p2.update(rewritten_test_df[['description', 'requirements']])
X_test_p2.reset_index(inplace=True)

# Combine updated text
X_test_p2['description'] = X_test_p2['description'].fillna('')
X_test_p2['requirements'] = X_test_p2['requirements'].fillna('')
X_test_p2['text'] = X_test_p2['description'] + " " + X_test_p2['requirements']

# 3. Transform using FROZEN Phase 1 TF-IDF Vectorizer (DO NOT FIT!)
X_test_tfidf_p2 = tfidf_p1.transform(X_test_p2['text'])

# 4. Predict using FROZEN Phase 1 Logistic Regression (DO NOT RETRAIN!)
y_pred_p2 = lr_p1.predict(X_test_tfidf_p2)
y_proba_p2 = lr_p1.predict_proba(X_test_tfidf_p2)[:, 1]

# 5. Evaluate Metrics & Degradation
acc_p2 = accuracy_score(y_test, y_pred_p2)
prec_p2 = precision_score(y_test, y_pred_p2)
rec_p2 = recall_score(y_test, y_pred_p2)
f1_p2 = f1_score(y_test, y_pred_p2)
auc_p2 = roc_auc_score(y_test, y_proba_p2)

rec_drop = rec_p1 - rec_p2
retention = (rec_p2 / rec_p1) * 100

print("\n" + "="*70)
print("📊 PHASE 2 RESULTS: LLM Text Evasion Attack")
print("="*70)
print(f"Accuracy         : {acc_p2:.4f}")
print(f"Precision        : {prec_p2:.4f}")
print(f"Recall           : {rec_p2:.4f}")
print(f"F1-Score         : {f1_p2:.4f}")
print(f"ROC-AUC          : {auc_p2:.4f}")
print("-" * 50)
print(f"Clean Recall (P1): {rec_p1:.4f} ({rec_p1*100:.2f}%)")
print(f"Attack Recall(P2): {rec_p2:.4f} ({rec_p2*100:.2f}%)")
print(f"Recall Drop      : {rec_drop:.4f} (-{rec_drop*100:.2f}%)")
print(f"Recall Retention : {retention:.2f}%")
print("\nClassification Report:\n", classification_report(y_test, y_pred_p2))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_p2))

In [ ]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    average_precision_score
)

print("=" * 70)
print("PHASE 3 — ADVERSARIAL RETRAINING")
print("=" * 70)


# ============================================================
# 1. LOAD AI-REWRITTEN TRAINING FRAUD JOBS
# ============================================================

PATH_TRAIN_REWRITTEN = (
    r'data/693_train_fraud_jobs_REWRITTEN.csv'
)

train_rewritten_fraud = load_csv_safe(PATH_TRAIN_REWRITTEN)

print(f"Rewritten fraud training jobs: {len(train_rewritten_fraud)}")


# ============================================================
# 2. CREATE ADVERSARIAL TRAINING DATA
# ============================================================

X_train_p3 = X_train.copy()

# Index using job_id
X_train_p3.set_index('job_id', inplace=True)

rewritten_train_df = train_rewritten_fraud.copy()
rewritten_train_df.set_index('job_id', inplace=True)

# Update ONLY description and requirements
X_train_p3.update(
    rewritten_train_df[['description', 'requirements']]
)

X_train_p3.reset_index(inplace=True)


# ============================================================
# 3. COMBINE TEXT
# ============================================================

X_train_p3['description'] = (
    X_train_p3['description'].fillna('')
)

X_train_p3['requirements'] = (
    X_train_p3['requirements'].fillna('')
)

X_train_p3['text'] = (
    X_train_p3['description']
    + " "
    + X_train_p3['requirements']
)


# ============================================================
# 4. CREATE NEW TF-IDF VECTORIZER
# ============================================================

print("\nFitting TF-IDF on adversarial training data...")

tfidf_p3 = TfidfVectorizer(
    max_features=3000,
    stop_words='english',
    ngram_range=(1, 2)
)

X_train_tfidf_p3 = tfidf_p3.fit_transform(
    X_train_p3['text']
)


# ============================================================
# 5. PREPARE CLEAN TEST TEXT
# ============================================================

X_test_clean_p3 = X_test.copy()

X_test_clean_p3['description'] = (
    X_test_clean_p3['description'].fillna('')
)

X_test_clean_p3['requirements'] = (
    X_test_clean_p3['requirements'].fillna('')
)

X_test_clean_p3['text'] = (
    X_test_clean_p3['description']
    + " "
    + X_test_clean_p3['requirements']
)


# ============================================================
# 6. PREPARE ATTACKED TEST TEXT
#    X_test_p2 was created in Phase 2
# ============================================================

X_test_attack_p3 = X_test_p2.copy()

X_test_attack_p3['description'] = (
    X_test_attack_p3['description'].fillna('')
)

X_test_attack_p3['requirements'] = (
    X_test_attack_p3['requirements'].fillna('')
)

X_test_attack_p3['text'] = (
    X_test_attack_p3['description']
    + " "
    + X_test_attack_p3['requirements']
)


# ============================================================
# 7. TRANSFORM BOTH TEST SETS
#    IMPORTANT: DO NOT FIT TF-IDF AGAIN
# ============================================================

X_test_tfidf_clean_p3 = tfidf_p3.transform(
    X_test_clean_p3['text']
)

X_test_tfidf_attack_p3 = tfidf_p3.transform(
    X_test_attack_p3['text']
)


# ============================================================
# 8. TRAIN NEW LOGISTIC REGRESSION MODEL
# ============================================================

print("Training adversarially retrained Logistic Regression...")

lr_p3 = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

lr_p3.fit(
    X_train_tfidf_p3,
    y_train
)


# ============================================================
# 9. FUNCTION FOR EVALUATION
# ============================================================

def evaluate_model(model, X_data, y_true, dataset_name):

    y_pred = model.predict(X_data)

    y_proba = model.predict_proba(X_data)[:, 1]

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(
        y_true, y_pred, zero_division=0
    )
    recall = recall_score(
        y_true, y_pred, zero_division=0
    )
    f1 = f1_score(
        y_true, y_pred, zero_division=0
    )
    roc_auc = roc_auc_score(
        y_true, y_proba
    )
    pr_auc = average_precision_score(
        y_true, y_proba
    )

    print("\n" + "=" * 70)
    print(f"PHASE 3 — {dataset_name}")
    print("=" * 70)

    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print(f"ROC-AUC   : {roc_auc:.4f}")
    print(f"PR-AUC    : {pr_auc:.4f}")

    print("\nClassification Report:")
    print(
        classification_report(
            y_true,
            y_pred,
            zero_division=0
        )
    )

    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "predictions": y_pred,
        "probabilities": y_proba
    }


# ============================================================
# 10. EVALUATE P3 ON CLEAN TEST DATA
# ============================================================

p3_clean = evaluate_model(
    lr_p3,
    X_test_tfidf_clean_p3,
    y_test,
    "CLEAN TEST DATA"
)


# ============================================================
# 11. EVALUATE P3 ON ATTACKED TEST DATA
# ============================================================

p3_attack = evaluate_model(
    lr_p3,
    X_test_tfidf_attack_p3,
    y_test,
    "LLM-REWRITTEN ATTACK TEST DATA"
)


# ============================================================
# 12. COMPARE CLEAN vs ATTACK
# ============================================================

clean_recall = p3_clean["recall"]
attack_recall = p3_attack["recall"]

recall_change = attack_recall - clean_recall

print("\n" + "=" * 70)
print("PHASE 3 — ROBUSTNESS ANALYSIS")
print("=" * 70)

print(
    f"P3 Clean Recall  : "
    f"{clean_recall:.4f} ({clean_recall * 100:.2f}%)"
)

print(
    f"P3 Attack Recall : "
    f"{attack_recall:.4f} ({attack_recall * 100:.2f}%)"
)

print(
    f"Recall Change    : "
    f"{recall_change:.4f} ({recall_change * 100:.2f} percentage points)"
)


# ============================================================
# 13. COMPARE PHASE 1 → PHASE 2 → PHASE 3
# ============================================================

print("\n" + "=" * 70)
print("OVERALL RESEARCH COMPARISON")
print("=" * 70)

print(
    f"Phase 1 Clean Recall : "
    f"{rec_p1:.4f} ({rec_p1 * 100:.2f}%)"
)

print(
    f"Phase 2 Attack Recall: "
    f"{rec_p2:.4f} ({rec_p2 * 100:.2f}%)"
)

print(
    f"Phase 3 Clean Recall : "
    f"{clean_recall:.4f} ({clean_recall * 100:.2f}%)"
)

print(
    f"Phase 3 Attack Recall: "
    f"{attack_recall:.4f} ({attack_recall * 100:.2f}%)"
)

In [ ]:
# Install dependencies before running this notebook:
# pip install -r requirements.txt

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

print("="*70)
print("PHASE 4 — METADATA-ONLY DEFENSE")
print("="*70)

# 1. Prepare Metadata Features from Original Dataset
X_meta = X.copy()

# Feature Engineering: Structural presence flags
X_meta['has_company_profile'] = X_meta['company_profile'].notna().astype(int)
X_meta['has_requirements'] = X_meta['requirements'].notna().astype(int)
X_meta['has_benefits'] = X_meta['benefits'].notna().astype(int)
X_meta['has_salary_range'] = X_meta['salary_range'].notna().astype(int)

cat_cols = ['employment_type', 'required_experience', 'required_education', 'industry', 'function']
num_cols = [
    'telecommuting', 'has_company_logo', 'has_questions',
    'has_company_profile', 'has_requirements', 'has_benefits', 'has_salary_range'
]

# Fill missing categorical values
for col in cat_cols:
    X_meta[col] = X_meta[col].fillna('Unspecified')

meta_features = cat_cols + num_cols

# Split exact train/test metadata
X_meta_train = X_meta.loc[X_train.index, meta_features]
X_meta_test = X_meta.loc[X_test.index, meta_features]

# 2. Build Metadata Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', 'passthrough', num_cols)
    ]
)

print("⚙️ Processing metadata features (encoding categoricals & passing binary flags)...")
X_train_meta_proc = preprocessor.fit_transform(X_meta_train)
X_test_meta_proc = preprocessor.transform(X_meta_test)

# Save processed test set for Phase 5
X_test_meta_raw = X_meta_test.copy()

# 3. Define Metadata Classifiers
meta_models = {
    "Meta_LogisticRegression": LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    "Meta_RandomForest": RandomForestClassifier(class_weight='balanced', random_state=42),
    "Meta_LightGBM": LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1),
    "Meta_CatBoost": CatBoostClassifier(auto_class_weights='Balanced', random_state=42, verbose=0)
}

# 4. Train & Evaluate Models
p4_results = []
trained_meta_models = {}

print("\n⚙️ Training Phase 4 Metadata Models...")
for name, model in meta_models.items():
    model.fit(X_train_meta_proc, y_train)
    trained_meta_models[name] = model
    
    y_pred = model.predict(X_test_meta_proc)
    y_proba = model.predict_proba(X_test_meta_proc)[:, 1] if hasattr(model, "predict_proba") else np.nan
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_proba)
    
    p4_results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "AUC": auc
    })

p4_df = pd.DataFrame(p4_results)

print("\n" + "="*70)
print("📊 PHASE 4 FINAL RESULTS TABLE (Clean Metadata)")
print("="*70)
print(p4_df.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

print("="*70)
print("PHASE 5 — STRUCTURAL METADATA SPOOFING ATTACK")
print("="*70)

# 1. Create a copy of the raw Phase 4 test metadata
X_test_spoofed = X_test_meta_raw.copy()

# 2. Identify the 173 test fraud indices
fraud_test_indices = y_test[y_test == 1].index
print(f"✓ Target set: {len(fraud_test_indices)} fraudulent test postings")

# 3. Apply Metadata Spoofing Attack on the 173 fraud postings
spoof_flags_to_1 = [
    'has_company_logo',
    'has_questions',
    'has_company_profile',
    'has_requirements',
    'has_benefits',
    'has_salary_range'
]

for flag in spoof_flags_to_1:
    if flag in X_test_spoofed.columns:
        X_test_spoofed.loc[fraud_test_indices, flag] = 1

if 'telecommuting' in X_test_spoofed.columns:
    X_test_spoofed.loc[fraud_test_indices, 'telecommuting'] = 0

print("✓ Structural binary flags spoofed to match legitimate corporate postings.")

# 4. Transform Spoofed Metadata using Phase 4 Preprocessor
X_test_spoofed_proc = preprocessor.transform(X_test_spoofed)

# 5. Clean Recalls from Phase 4 for Comparison
clean_recalls = {
    "Meta_LogisticRegression": p4_df.loc[p4_df['Model'] == "Meta_LogisticRegression", "Recall"].values[0],
    "Meta_RandomForest": p4_df.loc[p4_df['Model'] == "Meta_RandomForest", "Recall"].values[0],
    "Meta_LightGBM": p4_df.loc[p4_df['Model'] == "Meta_LightGBM", "Recall"].values[0],
    "Meta_CatBoost": p4_df.loc[p4_df['Model'] == "Meta_CatBoost", "Recall"].values[0]
}

# 6. Evaluate Trained Phase 4 Models on Spoofed Metadata
p5_results = []

for name, model in trained_meta_models.items():
    y_pred = model.predict(X_test_spoofed_proc)
    
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test_spoofed_proc)[:, 1]
        auc = roc_auc_score(y_test, y_proba)
    else:
        auc = np.nan

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    
    clean_rec = clean_recalls[name]
    rec_drop = clean_rec - rec
    retention = (rec / clean_rec) * 100 if clean_rec > 0 else 0
    
    p5_results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Clean Recall": clean_rec,
        "Spoofed Recall": rec,
        "Recall Drop": rec_drop,
        "Retention (%)": retention,
        "AUC": auc
    })

p5_df = pd.DataFrame(p5_results)

print("\n" + "="*70)
print("📊 PHASE 5 FINAL RESULTS TABLE (Spoofed Metadata)")
print("="*70)
print(p5_df[["Model", "Accuracy", "Precision", "Clean Recall", "Spoofed Recall", "Recall Drop", "Retention (%)", "AUC"]].to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar

print("="*70)
print("PHASE 6 — CORRECTED STATISTICAL VALIDATION (MCNEMAR'S TESTS)")
print("="*70)

# 1. Extract Exact Predictions from Trained LightGBM Models
lgbm_model = trained_meta_models["Meta_LightGBM"]
y_pred_p4_lgbm = lgbm_model.predict(X_test_meta_proc)
y_pred_p5_lgbm = lgbm_model.predict(X_test_spoofed_proc)

# 2. Automated McNemar Test Function
def run_mcnemar_analysis(y_true, pred_a, pred_b, name_a, name_b):
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)

    both_correct = np.sum(correct_a & correct_b)
    a_corr_b_wrong = np.sum(correct_a & ~correct_b)
    a_wrong_b_corr = np.sum(~correct_a & correct_b)
    both_wrong = np.sum(~correct_a & ~correct_b)

    table = np.array([
        [both_correct, a_corr_b_wrong],
        [a_wrong_b_corr, both_wrong]
    ])

    result = mcnemar(table, exact=False, correction=True)

    print("-" * 70)
    print(f"📊 MCNEMAR'S TEST: {name_a}  VS  {name_b}")
    print("-" * 70)
    print("Contingency Table (Overall Dataset, N = 3576):")
    print(f"  Both Correct                        : {both_correct}")
    print(f"  {name_a} Correct / {name_b} Wrong   : {a_corr_b_wrong}")
    print(f"  {name_a} Wrong / {name_b} Correct   : {a_wrong_b_corr}")
    print(f"  Both Wrong                          : {both_wrong}")
    print(f"\nChi-Square Statistic : {result.statistic:.4f}")
    print(f"p-value              : {result.pvalue:.6e}")

    if result.pvalue < 0.05:
        print("✅ Result: STATISTICALLY SIGNIFICANT difference (p < 0.05)")
    else:
        print("❌ Result: NOT statistically significant overall (p >= 0.05)")
    print("-" * 70 + "\n")
    return table, result

# --- EXECUTE STATISTICAL PAIR COMPARISONS ---

# Test 1: Baseline Text vs LLM Attack Text
table_1, res_1 = run_mcnemar_analysis(
    y_test, y_pred_p1, y_pred_p2,
    "P1 Clean Text", "P2 LLM Text Attack"
)

# Test 2: Clean Metadata Defense vs Spoofed Metadata Attack
table_2, res_2 = run_mcnemar_analysis(
    y_test, y_pred_p4_lgbm, y_pred_p5_lgbm,
    "P4 Clean Metadata (LightGBM)", "P5 Spoofed Metadata (LightGBM)"
)

# Test 3: LLM Attacked Text vs Clean Metadata Defense
table_3, res_3 = run_mcnemar_analysis(
    y_test, y_pred_p2, y_pred_p4_lgbm,
    "P2 LLM Attacked Text", "P4 Clean Metadata (LightGBM)"
)


# --- FINAL CORRECTED MASTER SUMMARY TABLE ---
master_summary_corrected = [
    {
        "Phase": "Phase 1: Baseline Text",
        "Model": "Logistic Regression (TF-IDF)",
        "Scenario / Condition": "Clean Original Data",
        "Fraud Recall": "85.55%",
        "Primary Key Finding": "Strong baseline text detection on standard clean data."
    },
    {
        "Phase": "Phase 2: LLM Text Attack",
        "Model": "Logistic Regression (TF-IDF)",
        "Scenario / Condition": "AI Text Rewriting Attack",
        "Fraud Recall": "60.69%",
        "Primary Key Finding": "Evasion Attack: LLM rewrites drop text recall by -24.86%."
    },
    {
        "Phase": "Phase 3: Retraining Attack",
        "Model": "Logistic Regression (TF-IDF)",
        "Scenario / Condition": "Adversarial Retraining",
        "Fraud Recall": "30.64%",
        "Primary Key Finding": "Catastrophic Degradation: Linguistic homogenization causes recall to drop further (-30.05%)."
    },
    {
        "Phase": "Phase 4: Metadata Defense",
        "Model": "Meta_LightGBM",
        "Scenario / Condition": "Clean Structural Metadata",
        "Fraud Recall": "84.97%",
        "Primary Key Finding": "Effective Secondary Defense: Non-textual metadata restores detection performance (84.97%)."
    },
    {
        "Phase": "Phase 5: Metadata Spoofing",
        "Model": "Meta_LightGBM",
        "Scenario / Condition": "Spoofed Structural Metadata",
        "Fraud Recall": "30.06%",
        "Primary Key Finding": "Structural Vulnerability: Spoofing binary flags crashes metadata recall to 30.06%."
    }
]

print("="*70)
print("🏆 FINAL CORRECTED MASTER EXPERIMENTAL SUMMARY TABLE")
print("="*70)
print(pd.DataFrame(master_summary_corrected).to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.contingency_tables import mcnemar

print("="*70)
print("PHASE 6 — STATISTICAL VALIDATION & MASTER SUMMARY")
print("="*70)

# 1. Extract Exact Predictions from Trained LightGBM Models
lgbm_model = trained_meta_models["Meta_LightGBM"]
y_pred_p4_lgbm = lgbm_model.predict(X_test_meta_proc)
y_pred_p5_lgbm = lgbm_model.predict(X_test_spoofed_proc)

# 2. Automated McNemar Test Function
def run_mcnemar_analysis(y_true, pred_a, pred_b, name_a, name_b):
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)

    both_correct = np.sum(correct_a & correct_b)
    a_corr_b_wrong = np.sum(correct_a & ~correct_b)
    a_wrong_b_corr = np.sum(~correct_a & correct_b)
    both_wrong = np.sum(~correct_a & ~correct_b)

    table = np.array([
        [both_correct, a_corr_b_wrong],
        [a_wrong_b_corr, both_wrong]
    ])

    result = mcnemar(table, exact=False, correction=True)

    print("-" * 70)
    print(f"📊 MCNEMAR'S TEST: {name_a}  VS  {name_b}")
    print("-" * 70)
    print("Contingency Table (Overall Dataset, N = 3576):")
    print(f"  Both Correct                        : {both_correct}")
    print(f"  {name_a} Correct / {name_b} Wrong   : {a_corr_b_wrong}")
    print(f"  {name_a} Wrong / {name_b} Correct   : {a_wrong_b_corr}")
    print(f"  Both Wrong                          : {both_wrong}")
    print(f"\nChi-Square Statistic : {result.statistic:.4f}")
    print(f"p-value              : {result.pvalue:.6e}")

    if result.pvalue < 0.05:
        print("✅ Result: STATISTICALLY SIGNIFICANT difference (p < 0.05)")
    else:
        print("❌ Result: NOT statistically significant overall (p >= 0.05)")
    print("-" * 70 + "\n")
    return table, result

# --- EXECUTE STATISTICAL PAIR COMPARISONS ---

# Test 1: Baseline Text vs LLM Attack Text
table_1, res_1 = run_mcnemar_analysis(
    y_test, y_pred_p1, y_pred_p2,
    "P1 Clean Text", "P2 LLM Text Attack"
)

# Test 2: Clean Metadata Defense vs Spoofed Metadata Attack
table_2, res_2 = run_mcnemar_analysis(
    y_test, y_pred_p4_lgbm, y_pred_p5_lgbm,
    "P4 Clean Metadata (LightGBM)", "P5 Spoofed Metadata (LightGBM)"
)

# Test 3: LLM Attacked Text vs Clean Metadata Defense
table_3, res_3 = run_mcnemar_analysis(
    y_test, y_pred_p2, y_pred_p4_lgbm,
    "P2 LLM Attacked Text", "P4 Clean Metadata (LightGBM)"
)

# --- FINAL CORRECTED MASTER SUMMARY TABLE ---
master_summary_corrected = [
    {
        "Phase": "Phase 1: Baseline Text",
        "Model": "Logistic Regression (TF-IDF)",
        "Scenario / Condition": "Clean Original Data",
        "Fraud Recall": "85.55%",
        "Primary Key Finding": "Strong baseline text detection on standard clean human scams."
    },
    {
        "Phase": "Phase 2: LLM Text Attack",
        "Model": "Logistic Regression (TF-IDF)",
        "Scenario / Condition": "AI Text Rewriting Attack",
        "Fraud Recall": "60.69%",
        "Primary Key Finding": "Evasion Attack: Zero-shot LLM rewrites drop text recall by -24.86%."
    },
    {
        "Phase": "Phase 3: Retrained Text",
        "Model": "Logistic Regression (TF-IDF)",
        "Scenario / Condition": "Adversarial Retraining",
        "Fraud Recall": "68.21% (Attack) / 57.80% (Clean)",
        "Primary Key Finding": "Trade-off: Recovers AI attack recall (+7.52%) but incurs clean generalization penalty (-27.75%)."
    },
    {
        "Phase": "Phase 4: Metadata Defense",
        "Model": "Meta_LightGBM",
        "Scenario / Condition": "Clean Structural Metadata",
        "Fraud Recall": "84.97%",
        "Primary Key Finding": "Effective Secondary Defense: Non-textual structural flags restore recall to 84.97%."
    },
    {
        "Phase": "Phase 5: Metadata Spoofing",
        "Model": "Meta_LightGBM",
        "Scenario / Condition": "Spoofed Structural Metadata",
        "Fraud Recall": "30.06%",
        "Primary Key Finding": "Structural Fragility: Toggling binary profile flags crashes metadata recall by -54.91%."
    }
]

print("="*70)
print("🏆 FINAL CORRECTED MASTER EXPERIMENTAL SUMMARY TABLE")
print("="*70)
summary_df = pd.DataFrame(master_summary_corrected)
pd.set_option('display.max_colwidth', None)
print(summary_df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Set global publication style
sns.set_theme(style="whitegrid")
plt.rcParams.update(
    {
        "font.size": 11,
        "font.family": "sans-serif",
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "figure.titlesize": 15,
    }
)

# ==============================================================================
# FIGURE 1: CLEAN & UNCLUTTERED FRAUD RECALL PROGRESSION
# ==============================================================================
fig, ax = plt.subplots(figsize=(11, 6.5), dpi=300)

phases = [
    "P1: Clean Text\n(Baseline)",
    "P2: LLM Text\n(Attack)",
    "P3: Retrained Text\n(Attack Set)",
    "P4: Clean Meta\n(Defense)",
    "P5: Spoofed Meta\n(Attack)",
]
recalls = [85.55, 60.69, 68.21, 84.97, 30.06]
colors = ["#2b5c8f", "#d95f02", "#7570b3", "#2ca02c", "#d62728"]

bars = ax.bar(
    phases, recalls, color=colors, width=0.55, edgecolor="black", linewidth=0.8
)

# Add value labels directly above bars
for bar in bars:
    yval = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.0,
        yval + 1.5,
        f"{yval:.2f}%",
        ha="center",
        va="bottom",
        fontweight="bold",
        fontsize=10.5,
    )

ax.set_ylabel("Fraud Class Recall (%)", fontweight="bold")
ax.set_ylim(0, 112)  # Expanded headroom to prevent annotation collision
ax.set_title(
    "Figure 1: Fraud Detection Recall Across Experimental Phases",
    fontweight="bold",
    pad=15,
)

# Annotation 1: LLM Evasion Drop (Positioned in upper open space)
ax.annotate(
    "LLM Evasion\n(-24.86 pp)",
    xy=(0.95, 63),
    xycoords="data",
    xytext=(0.5, 96),
    textcoords="data",
    arrowprops=dict(
        arrowstyle="->",
        color="#d95f02",
        lw=1.8,
        shrinkA=0,
        shrinkB=3,
        connectionstyle="arc3,rad=-0.18",
    ),
    ha="center",
    va="bottom",
    fontweight="bold",
    color="#d95f02",
    fontsize=10,
)

# Annotation 2: Metadata Spoofing Drop (Positioned in upper open space)
ax.annotate(
    "Spoofing Attack\n(-54.91 pp)",
    xy=(3.95, 33),
    xycoords="data",
    xytext=(3.5, 96),
    textcoords="data",
    arrowprops=dict(
        arrowstyle="->",
        color="#d62728",
        lw=1.8,
        shrinkA=0,
        shrinkB=3,
        connectionstyle="arc3,rad=-0.18",
    ),
    ha="center",
    va="bottom",
    fontweight="bold",
    color="#d62728",
    fontsize=10,
)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams.update(
    {
        "font.size": 11,
        "font.family": "sans-serif",
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "figure.titlesize": 15,
    }
)

# ==============================================================================
# FIGURE 4: LIGHTGBM METADATA FEATURE IMPORTANCE
# ==============================================================================
# Extract feature names and importances from your trained LightGBM model
feature_names = preprocessor.get_feature_names_out()
importances = trained_meta_models["Meta_LightGBM"].feature_importances_

# Clean up feature names for clean publication display
clean_names = [
    f.replace("cat__", "").replace("num__", "").replace("_", " ").title()
    for f in feature_names
]

feat_df = (
    pd.DataFrame({"Feature": clean_names, "Importance": importances})
    .sort_values(by="Importance", ascending=False)
    .head(10)
)

fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
bars = ax.barh(
    feat_df["Feature"][::-1],
    feat_df["Importance"][::-1],
    color="#2ca02c",
    edgecolor="black",
    height=0.6,
)

ax.set_xlabel("Feature Importance (Split Count)", fontweight="bold")
ax.set_title(
    "Figure 4: Top 10 Metadata Features Driving Fraud Classification",
    fontweight="bold",
    pad=15,
)

for bar in bars:
    width = bar.get_width()
    ax.text(
        width + 5,
        bar.get_y() + bar.get_height() / 2,
        f"{int(width)}",
        ha="left",
        va="center",
        fontweight="bold",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

# ==============================================================================
# FIGURE 5: COMPREHENSIVE MULTI-METRIC PERFORMANCE HEATMAP
# ==============================================================================
metrics_data = {
    "Precision": [0.4485, 0.3659, 0.5784, 0.3973, 0.1891],
    "Recall": [0.8555, 0.6069, 0.6821, 0.8497, 0.3006],
    "F1-Score": [0.5885, 0.4565, 0.6260, 0.5414, 0.2321],
    "ROC-AUC": [0.9648, 0.8862, 0.9619, 0.9620, 0.7183],
}

phase_labels = [
    "P1: Baseline Text",
    "P2: LLM Text Attack",
    "P3: Retrained Text",
    "P4: Clean Metadata",
    "P5: Spoofed Metadata",
]

df_metrics = pd.DataFrame(metrics_data, index=phase_labels)

fig, ax = plt.subplots(figsize=(9, 5), dpi=300)
sns.heatmap(
    df_metrics,
    annot=True,
    fmt=".2%",
    cmap="YlGnBu",
    cbar=True,
    ax=ax,
    linewidths=1,
    linecolor="white",
    annot_kws={"size": 11, "weight": "bold"},
)

ax.set_title(
    "Figure 5: Complete Cross-Phase Performance Matrix",
    fontweight="bold",
    pad=15,
)
plt.xticks(fontweight="bold")
plt.yticks(fontweight="bold", rotation=0)

plt.tight_layout()
plt.show()